In [9]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib as mpl

In [10]:
def setup_plot_style():
    """Configura stile matplotlib per paper"""
    plt.rcParams.update({
        'font.size': 14,
        'axes.labelsize': 16,
        'axes.titlesize': 16,
        'xtick.labelsize': 14,
        'ytick.labelsize': 14,
        'legend.fontsize': 14,
        'font.family': 'sans-serif',
        'font.sans-serif': ['Arial', 'DejaVu Sans'],
        'axes.linewidth': 1.5,
        'grid.linewidth': 0.8,
        'lines.linewidth': 2.5,
        'lines.markersize': 8,
        'xtick.major.width': 1.5,
        'ytick.major.width': 1.5,
        'figure.dpi': 300
    })


In [11]:
def layer_to_depth_percentage(layer_idx, n_layers):
    """Converte layer index a percentuale di profondità"""
    return (layer_idx / (n_layers - 1)) * 100


In [12]:
def plot_pseudotime_comparison(csv_files, model_names, output_path, 
                               metric='Pseudotime_Corr', ylabel=None,
                               figsize=(10, 6)):
    """
    Plot comparativo tra modelli per metriche di pseudotime
    
    Args:
        csv_files: lista di path ai CSV
        model_names: nomi dei modelli (per legenda)
        output_path: dove salvare il plot
        metric: colonna da plottare
        ylabel: label asse y (default: nome metrica)
        figsize: dimensioni figura
    """
    setup_plot_style()
    
    fig, ax = plt.subplots(figsize=figsize)
    
    # Palette tab10 (10 colori distinti)
    base_colors = plt.cm.tab10(np.arange(len(csv_files)))
    
    # Modifico il secondo colore se è grigio (indice 1)
    # Ad esempio assegno tab:orange
    colors = base_colors.copy()
    if len(csv_files) > 1:
        colors[1] = mpl.colors.to_rgba('tab:orange')
    
    for i, (csv_file, model_name) in enumerate(zip(csv_files, model_names)):
        df = pd.read_csv(csv_file)
        
        if 'layer_idx' in df.columns:
            layer_idx = df['layer_idx'].values
        else:
            layer_idx = df['Layer'].str.extract(r'(\d+)')[0].astype(int).values
        
        n_layers = layer_idx.max() + 1
        depth_pct = layer_to_depth_percentage(layer_idx, n_layers)
        
        ax.plot(depth_pct, df[metric], 
                marker='o', label=model_name, 
                color=colors[i], linewidth=2.0, markersize=7,
                linestyle='--')  # linee tratteggiate
    
    ax.set_xlabel('Layer Depth Percentage', fontsize=16)
    ax.set_ylabel(ylabel or metric.replace('_', ' '), fontsize=16)
    ax.grid(True, alpha=0.3, linewidth=0.8)
    ax.legend(loc='lower right', frameon=True, fancybox=False, edgecolor='black')
    ax.set_xlim(-5, 105)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Salvato: {output_path}")


In [13]:
plot_pseudotime_comparison(
    csv_files=[
        
        '/data2/home/vcivale/scfm-layer-analysis-refactored/data/pseudotime_results/GSE276896_adata_meta_tahoe_1b_embeddings_results.csv',
        '/data2/home/vcivale/scfm-layer-analysis-refactored/data/pseudotime_results/GSE276896_adata_meta_scfoundation_embeddings_results.csv'
    ],
    model_names=['Tahoe X1', 'scFoundation'],
    output_path='/data2/home/vcivale/scfm-layer-analysis-refactored/figures/GSE276896_pseudotime_comparison.pdf',
    metric='Pseudotime_Corr',
    ylabel='Pseudotime Correlation'
)

✓ Salvato: /data2/home/vcivale/scfm-layer-analysis-refactored/figures/GSE276896_pseudotime_comparison.pdf


In [14]:
def plot_semantic_similarity_multi_datasets(csv_files_list, model_names, dataset_names, output_path, figsize=(18, 6)):
    """
    Plot comparativo di semantic similarity per due modelli su 3 dataset affiancati.
    
    Args:
        csv_files_list: lista di liste di CSV. Ogni elemento è lista di 2 CSV per i due modelli su un dataset.
                       Quindi [[csv_model1_dataset1, csv_model2_dataset1], [csv_model1_dataset2, csv_model2_dataset2], ...]
        model_names: lista con 2 nomi modelli (es. ['Tahoe-X1', 'scFoundation'])
        dataset_names: lista con 3 nomi dataset, mostrati come titoli sopra ogni plot
        output_path: path per salvare la figura
        figsize: dimensioni della figura complessiva
    """
    setup_plot_style()
    
    n_datasets = len(csv_files_list)
    assert len(model_names) == 2, "La funzione è fatta per confrontare 2 modelli."
    assert len(dataset_names) == n_datasets, "Numero nomi dataset deve corrispondere al numero di dataset."
    
    fig, axes = plt.subplots(1, n_datasets, figsize=figsize, sharey=True)
    if n_datasets == 1:
        axes = [axes]
    
    # Palette tab10 come nella funzione pseudotime
    base_colors = plt.cm.tab10(np.arange(2))
    colors = base_colors.copy()
    colors[1] = mpl.colors.to_rgba('tab:orange')
    
    for i, ax in enumerate(axes):
        csv_files = csv_files_list[i]
        dataset_name = dataset_names[i]
        
        for j, (csv_file, model_name) in enumerate(zip(csv_files, model_names)):
            df = pd.read_csv(csv_file)
            layer_idx = df['layer'].values
            n_layers = layer_idx.max() + 1
            depth_pct = layer_to_depth_percentage(layer_idx, n_layers)
            
            ax.plot(depth_pct, df['correlation'], 
                    marker='o', label=model_name if i == n_datasets-1 else '',  # legenda solo nell'ultimo plot
                    color=colors[j], linewidth=2.0, markersize=7, linestyle='--')
        
        ax.set_xlabel('Layer Depth Percentage', fontsize=14)
        if i == 0:
            ax.set_ylabel('Spearman Correlation', fontsize=14)
        ax.set_title(dataset_name, fontsize=14)  # stile coerente con grafico
        
        ax.grid(True, alpha=0.3, linewidth=0.8)
        ax.set_xlim(-5, 105)
    
    # Legenda solo nell'ultimo subplot, in basso a destra
    axes[-1].legend(loc='lower right', frameon=True, fancybox=False, edgecolor='black')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✓ Saved plot at: {output_path}")


In [15]:
csv_files_list = [
    ['data/perturbation_metrics/D1_Rest.assigned_guide_undersampled_tahoe_1b_semantic_similarity.csv', 'data/perturbation_metrics/D1_Rest.assigned_guide_undersampled_scfoundation_embeddings_semantic_similarity.csv'],
    ['data/perturbation_metrics/D1_Stim8hr.assigned_guide_undersampled_tahoe_1b_semantic_similarity.csv', 'data/perturbation_metrics/D1_Stim8hr.assigned_guide_undersampled_scfoundation_embeddings_semantic_similarity.csv'],
    ['data/perturbation_metrics/D1_Stim48hr.assigned_guide_undersampled_tahoe_1b_semantic_similarity.csv', 'data/perturbation_metrics/D1_Stim48hr.assigned_guide_undersampled_scfoundation_embeddings_semantic_similarity.csv'],
]

model_names = ['Tahoe-X1', 'scFoundation']
dataset_names = ['D1 Rest', 'D1 Stim8hr', 'D1 Stim48hr']

plot_semantic_similarity_multi_datasets(csv_files_list, model_names, dataset_names, 'figures/D1_perturbation_comparison.pdf')


✓ Saved plot at: figures/D1_perturbation_comparison.pdf
